# Policy Exploration Notebook

Compare tax policy scenarios using the macro-tax-model library.

**Run Streamlit app for interactive sliders:**
```
streamlit run app.py
```

Use this notebook for deeper analysis, custom scenarios, and sensitivity studies.

In [ ]:
import sys; sys.path.insert(0, '..')

from tax_model import (
    Calibration, Economy, TaxPolicy,
    LaborIncomeTax, ConsumptionTax, LandValueTax,
    CorporateTax, PigouvianTax, CapitalGainsTax,
)
from tax_model.results import PolicyComparison
from tax_model.visualization import plot_dashboard, plot_macro_bars, plot_incidence
from tax_model.sensitivity import sweep, monte_carlo, credible_interval
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load calibration and build economy
cal = Calibration.from_yaml('../calibration/us_2024.yaml')
economy = Economy(cal)

# Approximate current US law
CURRENT_LAW_BRACKETS = [(0.0, 0.10), (0.5, 0.22), (1.5, 0.24), (3.0, 0.32), (7.0, 0.37)]

current_law = TaxPolicy(
    label='Current Law',
    labor_income=LaborIncomeTax(brackets=CURRENT_LAW_BRACKETS),
    corporate=CorporateTax(rate=0.21),
    capital_gains=CapitalGainsTax(rate=0.20, lock_in_discount=0.30),
)

baseline = economy.solve(current_law)
print(baseline.summary())

## Scenario 1: Progressive Consumption Tax (X-Tax style)

In [ ]:
consumption_tax = TaxPolicy(
    label='Progressive Consumption Tax',
    labor_income=LaborIncomeTax(brackets=[(0.0, 0.0)]),  # wage tax removed
    consumption=ConsumptionTax(rate=0.15, prebate_fraction=0.80),
    corporate=CorporateTax(rate=0.15, immediate_expensing=True),
    capital_gains=CapitalGainsTax(rate=0.0),
)

reform1 = economy.solve(consumption_tax)
comp1 = PolicyComparison(baseline=baseline, reform=reform1)
print(comp1.summary())

In [ ]:
fig = plot_dashboard(comp1)
plt.show()

## Scenario 2: Land Value Tax swap (LVT replaces corporate tax)

In [ ]:
lvt_swap = TaxPolicy(
    label='LVT swap (replaces corp tax)',
    labor_income=LaborIncomeTax(brackets=CURRENT_LAW_BRACKETS),
    corporate=CorporateTax(rate=0.0),   # corporate tax eliminated
    land_value=LandValueTax(rate=0.04), # ~revenue-neutral LVT
    capital_gains=CapitalGainsTax(rate=0.20, lock_in_discount=0.30),
)

reform2 = economy.solve(lvt_swap)
comp2 = PolicyComparison(baseline=baseline, reform=reform2)
print(comp2.summary())
fig = plot_dashboard(comp2)
plt.show()

## Scenario 3: Carbon fee + full dividend recycling

In [ ]:
carbon_fee = TaxPolicy(
    label='$100/ton carbon fee + dividend',
    labor_income=LaborIncomeTax(brackets=CURRENT_LAW_BRACKETS),
    corporate=CorporateTax(rate=0.21),
    capital_gains=CapitalGainsTax(rate=0.20, lock_in_discount=0.30),
    pigouvian=PigouvianTax(rate_per_unit=100.0, dividend_recycling_fraction=1.0),
)

reform3 = economy.solve(carbon_fee)
comp3 = PolicyComparison(baseline=baseline, reform=reform3)
print(comp3.summary())
fig = plot_dashboard(comp3)
plt.show()

## Sensitivity analysis: How sensitive is GDP effect to labor supply elasticity?

In [ ]:
from tax_model.sensitivity import sweep
from tax_model.visualization import plot_sensitivity_tornado

frisch_values = np.linspace(0.2, 0.9, 8)
comps = sweep(
    cal, current_law, consumption_tax,
    parameter_path='households.frisch_elasticity',
    values=list(frisch_values),
)

gdp_changes = [c.gdp_change_pct for c in comps]
plt.figure(figsize=(7, 4))
plt.plot(frisch_values, gdp_changes, 'o-', color='#2166AC')
plt.xlabel('Frisch elasticity of labor supply')
plt.ylabel('GDP change (%)')
plt.title('GDP effect of progressive consumption tax\nvs. Frisch elasticity assumption')
plt.axhline(0, color='gray', linewidth=0.8)
plt.tight_layout()
plt.show()

## Monte Carlo: Credible intervals over uncertain parameters

In [ ]:
from tax_model.sensitivity import monte_carlo, credible_interval
from tax_model.visualization import plot_monte_carlo

mc_results = monte_carlo(cal, current_law, consumption_tax, n_draws=300, seed=42)

for metric in ['gdp_change_pct', 'capital_change_pct', 'wage_change_pct']:
    ci = credible_interval(mc_results, metric)
    print(f"{metric:25s}: {ci['mean']:+.2f}%  (90% CI: {ci['lower']:+.2f}% to {ci['upper']:+.2f}%)")

fig = plot_monte_carlo(mc_results)
plt.show()